In [3]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')
from scipy.linalg import cho_factor, cho_solve
from google.colab import drive

drive.mount('/content/drive')
path = "/content/drive/My Drive/Campbell A data/preprocess_data.parquet"

t0 = time.time()

df = pd.read_parquet(path)
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(['DATE', 'permno']).reset_index(drop=True)
print(f"  Shape: {df.shape}  |  "
      f"{df['DATE'].min().date()} to {df['DATE'].max().date()}  |  "
      f"{df['permno'].nunique():,} stocks  |  loaded in {time.time()-t0:.1f}s")

TARGET     = 'exret'
MACRO_COLS = ['tbl', 'd/p', 'e/p', 'b/m', 'tms', 'dfy', 'ntis', 'svar']
CHAR_COLS  = [
    'mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom',
    'mom1m', 'mom6m', 'mom12m', 'mom36m', 'pricedelay', 'turn',
    'absacc', 'acc', 'age', 'agr', 'bm', 'bm_ia', 'cashdebt', 'cashpr',
    'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chinv', 'chpmia',
    'convind', 'currat', 'depr', 'divi', 'divo', 'dy', 'egr', 'ep',
    'gma', 'grcapx', 'grltnoa', 'herf', 'hire', 'invest', 'lev', 'lgr',
    'mve_ia', 'operprof', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr',
    'pchgm_pchsale', 'pchquick', 'pchsale_pchinvt', 'pchsale_pchrect',
    'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'ps', 'quick', 'rd',
    'rd_mve', 'rd_sale', 'realestate', 'roic', 'salecash', 'saleinv',
    'salerec', 'secured', 'securedind', 'sgr', 'sin', 'sp', 'tang', 'tb',
    'aeavol', 'cash', 'chtx', 'cinvest', 'ear', 'nincr', 'roaq', 'roavol',
    'roeq', 'rsup', 'stdacc', 'stdcf', 'ms', 'baspread', 'ill', 'maxret',
    'retvol', 'std_dolvol', 'std_turn', 'zerotrade']
CHAR_COLS  = [c for c in CHAR_COLS  if c in df.columns]
MACRO_COLS = [c for c in MACRO_COLS if c in df.columns]
HAS_SIC2   = 'sic2' in df.columns
print(f"  Chars: {len(CHAR_COLS)}  |  Macros: {len(MACRO_COLS)}  |  SIC2: {HAS_SIC2}")

# Feature matrix
t1 = time.time()
REQUIRED = CHAR_COLS + MACRO_COLS + [TARGET]
df = df.dropna(subset=REQUIRED).reset_index(drop=True)
print(f"  Clean rows: {len(df):,}")

chars  = df[CHAR_COLS].values.astype(np.float32)
macros = df[MACRO_COLS].values.astype(np.float32)

# interactions via einsum (less memory)
interactions = np.einsum('ij,ik->ijk', chars, macros).reshape(
    len(df), len(CHAR_COLS) * len(MACRO_COLS))

X_base = np.hstack([chars, interactions])

if HAS_SIC2:
    sic2_dummies = pd.get_dummies(
        df['sic2'], prefix='sic2', drop_first=False
    ).values.astype(np.float32)
    X_all = np.hstack([X_base, sic2_dummies])
else:
    X_all = X_base

y_all   = df[TARGET].values.astype(np.float64)
years   = df['DATE'].dt.year.values
permnos = df['permno'].values
dates   = df['DATE'].values
mvel1   = df['mvel1'].values if 'mvel1' in df.columns else None

P = X_all.shape[1]
REG = 1e-8 * np.eye(P)

print(f"  Feature matrix: {X_all.shape}  ({X_all.nbytes/1e9:.2f} GB)  "
      f"built in {time.time()-t1:.1f}s")


# Huber IRLS with Cholesky
def huber_weights(residuals: np.ndarray, xi: float) -> np.ndarray:
    abs_res = np.abs(residuals)
    w = np.where(abs_res <= xi, 1.0, xi / np.maximum(abs_res, 1e-10))
    return w


def fit_huber_irls_chol(XtX_init: np.ndarray, Xty_init: np.ndarray,
                        X: np.ndarray, y: np.ndarray,
                        xi: float, theta_init: np.ndarray,
                        max_iter: int = 30, tol: float = 1e-5) -> np.ndarray:

    theta = theta_init.copy()
    for _ in range(max_iter):
        residuals = y - X @ theta
        w = huber_weights(residuals, xi)
        Xw    = X * w[:, np.newaxis]
        XtWX  = Xw.T @ X + REG
        XtWy  = Xw.T @ y
        # Cholesky instead of general solve (XtWX is PD)
        c, low = cho_factor(XtWX, lower=False, check_finite=False)
        theta_new = cho_solve((c, low), XtWy, check_finite=False)
        if np.max(np.abs(theta_new - theta)) < tol:
            theta = theta_new
            break
        theta = theta_new
    return theta


# OOS R2
def oos_r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    ss_res = np.dot(y_true - y_pred, y_true - y_pred)
    ss_tot = np.dot(y_true, y_true)
    return 1.0 - ss_res / ss_tot


# Expanding-window
VALIDATION_END_YEAR = 1986
TEST_END_YEAR       = 2016

all_test_years = sorted(
    set(years[(years > VALIDATION_END_YEAR) & (years <= TEST_END_YEAR)])
)
print(f"\nRunning expanding-window OLS+H  "
      f"({all_test_years[0]}–{all_test_years[-1]})...")

t2 = time.time()

pred_dates   = []
pred_permnos = []
pred_y_true  = []
pred_y_pred  = []
pred_mvel1   = []

theta = None

# accumulate XᵀX and Xᵀy incrementally.
X_f64 = X_all.astype(np.float64)
seed_mask  = years <= (all_test_years[0] - 1)
X_seed     = X_f64[seed_mask]
y_seed     = y_all[seed_mask]
XtX_accum  = X_seed.T @ X_seed      # (P, P)
Xty_accum  = X_seed.T @ y_seed      # (P,)

for year in all_test_years:
    train_mask = years <= (year - 1)
    test_mask  = years == year

    if year > all_test_years[0]:
        prev_mask = years == (year - 1)
        X_prev    = X_f64[prev_mask]
        y_prev    = y_all[prev_mask]
        XtX_accum += X_prev.T @ X_prev
        Xty_accum += X_prev.T @ y_prev

    X_test = X_f64[test_mask]
    y_test = y_all[test_mask]

    # warm-start
    if theta is None:
        c, low = cho_factor(XtX_accum + REG, lower=False, check_finite=False)
        theta  = cho_solve((c, low), Xty_accum, check_finite=False)

    # Huber threshold
    X_train = X_f64[train_mask]
    y_train = y_all[train_mask]
    resid_init = y_train - X_train @ theta
    xi = np.percentile(np.abs(resid_init), 99.9)

    theta = fit_huber_irls_chol(
        XtX_accum, Xty_accum,
        X_train, y_train,
        xi=xi, theta_init=theta)

    y_pred = X_test @ theta

    pred_dates.append(dates[test_mask])
    pred_permnos.append(permnos[test_mask])
    pred_y_true.append(y_test)
    pred_y_pred.append(y_pred)
    if mvel1 is not None:
        pred_mvel1.append(mvel1[test_mask])

    print(f"  {year}  |  train: {train_mask.sum():,}  |  "
          f"test: {test_mask.sum():,}  |  xi: {xi:.4f}  |  "
          f"elapsed: {time.time()-t2:.1f}s")

print(f"Loop finished in {time.time()-t2:.1f}s")

# Results
y_true_all = np.concatenate(pred_y_true)
y_pred_all = np.concatenate(pred_y_pred)
dates_all  = np.concatenate(pred_dates)
perm_all   = np.concatenate(pred_permnos)

r2_all = oos_r2(y_true_all, y_pred_all)
print(f"\n{'═'*65}")
print(f"  {'Subsample':<32}  {'OOS R2':>10}  {'Paper':>10}")
print(f"{'─'*65}")
print(f"  {'All stocks (panel)':<32}  {r2_all*100:>+10.4f}%  {'–3.46%':>10}")

if mvel1 is not None:
    mv_all     = np.concatenate(pred_mvel1)
    results_df = pd.DataFrame({
        'DATE':   dates_all,
        'mvel1':  mv_all,
        'y_true': y_true_all,
        'y_pred': y_pred_all,
    })

    for label, largest, paper_val in [
        ('Top 1000 (largest)',     True,  -11.28),
        ('Bottom 1000 (smallest)', False,  -1.30),
    ]:
        rows = []
        for _, grp in results_df.groupby('DATE', sort=False):
            idx = np.argsort(grp['mvel1'].values)
            idx = idx[-1000:] if largest else idx[:1000]
            rows.append(grp.iloc[idx])
        sub = pd.concat(rows, ignore_index=True)
        r2  = oos_r2(sub['y_true'].values, sub['y_pred'].values)
        print(f"  {label:<32}  {r2*100:>+10.4f}%  {paper_val:>+10.2f}%")

print(f"{'═'*65}")
print(f"\nTotal wall time: {time.time()-t0:.1f}s")

out = pd.DataFrame({
    'DATE':   dates_all,
    'permno': perm_all,
    'y_true': y_true_all,
    'y_pred': y_pred_all,
})
out_path = '/content/drive/My Drive/ols_predictions.csv'
out.to_csv(out_path, index=False)
print(f"Predictions saved → {out_path}  ({len(out):,} rows)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Shape: (3712808, 109)  |  1957-04-30 to 2016-12-30  |  29,825 stocks  |  loaded in 44.0s
  Chars: 94  |  Macros: 8  |  SIC2: True
  Clean rows: 3,712,808
  Feature matrix: (3712808, 920)  (13.66 GB)  built in 20.3s

Running expanding-window OLS+H  (1987–2016)...
  1987  |  train: 1,236,775  |  test: 82,404  |  xi: 1.0504  |  elapsed: 81.3s
  1988  |  train: 1,319,179  |  test: 83,415  |  xi: 1.0710  |  elapsed: 165.6s
  1989  |  train: 1,402,594  |  test: 81,216  |  xi: 1.0819  |  elapsed: 227.7s
  1990  |  train: 1,483,810  |  test: 80,207  |  xi: 1.0957  |  elapsed: 297.1s
  1991  |  train: 1,564,017  |  test: 79,274  |  xi: 1.1200  |  elapsed: 371.1s
  1992  |  train: 1,643,291  |  test: 80,972  |  xi: 1.1867  |  elapsed: 449.7s
  1993  |  train: 1,724,263  |  test: 86,150  |  xi: 1.2212  |  elapsed: 532.1s
  1994  |  train: 1,810,413  |  test: 95,088  |